<div style="font-family: 'Helvetica Neue', -apple-system, Arial, sans-serif;">

### Playground to Test: Do Signals have Predictive Power?
___

</div>

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

from modules.signals.fetch import get_sp500_universe, fetch_price_data, START_DATE, END_DATE
from modules.signals.momentum import compute_momentum
from modules.signals.mean_reversion import compute_mean_reversion
from modules.signals.testing import evaluate_signal

import pandas as pd

In [ ]:
from modules.signals.fetch import save_price_data

universe = get_sp500_universe()
print(f"{len(universe)} tickers across {universe['Sector'].nunique()} sectors")
close, volume = fetch_price_data(universe["Symbol"].tolist(), start=START_DATE, end=END_DATE)

failed_tickers = close.columns[close.isna().all()].tolist()
if failed_tickers:
    close = close.drop(columns=failed_tickers)
    volume = volume.drop(columns=failed_tickers)

save_price_data(universe, close, volume)
print(f"Close shape: {close.shape}")

# once the data's been saved...

# from modules.signals.fetch import load_price_data

# universe, close, volume = load_price_data()
# print(f"Close shape: {close.shape}")
# close.tail()

Fetching batch 1 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 2 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 3 / 13 (40 tickers)...


...

Fetching batch 12 / 13 (40 tickers)...


[*********************100%***********************]  40 of 40 completed


Fetching batch 13 / 13 (23 tickers)...


[*********************100%***********************]  23 of 23 completed


Saved universe, close, and volume to /Users/paulgrajzl/Documents/Quant-Projects/systematic-portfolio-engine-equities-1/data/
Close shape: (1907, 503)


In [5]:
momentum_signal = compute_momentum(close)
mean_reversion_signal = compute_mean_reversion(close)

print(f"Momentum shape: {momentum_signal.shape}, non-null: {momentum_signal.notna().sum().sum()}")
print(f"Mean Reversion shape: {mean_reversion_signal.shape}, non-null: {mean_reversion_signal.notna().sum().sum()}")

Momentum shape: (1907, 503), non-null: 907326
Mean Reversion shape: (1907, 503), non-null: 936240


In [6]:
HORIZON = 20  # arbitrary starting point, just to confirm the pipeline runs

mom_ic_series, mom_summary = evaluate_signal(momentum_signal, close, horizon=HORIZON)
print(pd.Series(mom_summary).round(4))

Mean IC                -0.0147
IC Std                  0.1954
Information Ratio      -0.0754
% Positive IC           0.5126
N Observations       1824.0000
dtype: float64


In [7]:
mr_ic_series, mr_summary = evaluate_signal(mean_reversion_signal, close, horizon=HORIZON)
print(pd.Series(mr_summary).round(4))

Mean IC                 0.0077
IC Std                  0.1766
Information Ratio       0.0437
% Positive IC           0.5138
N Observations       1882.0000
dtype: float64
